### __Import__

In [ ]:
from data_model_loader import *

model = load_model()
images = load_coco_2014_dataset()

# load file paths
with open('config.json', 'r') as file:
    config = json.load(file)
COCO_FOLDER = config['coco_folder']
ANNOTATION_FILE_PATH =  config['annotation_file_path']
MODEL_NAME = "base_model"
RESULTS_DIR = 'results/'


OD model already present locally.
data/coco2014\val2014.zip already exists. Skipping download.
data/coco2014\val2014 already exists. Skipping extraction.
data/coco2014\annotations_trainval2014.zip already exists. Skipping download.
data/coco2014\annotations_trainval2014 already exists. Skipping extraction.
COCO 2014 dataset download and extraction complete.
Randomly selecting 1500 pictures ..
Done!


### __Predict__

In [4]:
import base_model

results, inference_times = base_model.predict(model, images, n_images=10, data_path= COCO_FOLDER)

Inferencing images: 100%|██████████| 10/10 [00:02<00:00,  4.59it/s]


### __Store Results__

In [5]:
RESULTS_DIR = 'results/'

def store_results(results, results_dir):
    # create dir
    results_dir = pathlib.Path(results_dir)
    results_dir.mkdir(exist_ok=True, parents=True)
    results_file = results_dir/f"{MODEL_NAME}_results.json"

    # store results
    with open(results_file, 'w') as json_file:
        json.dump(results, json_file, indent=4)

store_results(results, RESULTS_DIR)

### __Evaluate__

In [6]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

annType = 'bbox'
results_file_path =  RESULTS_DIR + MODEL_NAME + "_results.json"
#initialize COCO ground truth api
cocoGt=COCO(ANNOTATION_FILE_PATH)

#initialize COCO detections api
cocoDt=cocoGt.loadRes(results_file_path)

# prepare ids
imgIds=sorted(cocoGt.getImgIds())
#imgIds=imgIds[0:100]
#imgId = imgIds[np.random.randint(100)]

# evaluate
cocoEval = COCOeval(cocoGt, cocoDt, annType)
cocoEval.params.imgIds = imgIds
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

# average precision scores
ap_scores = cocoEval.stats

loading annotations into memory...
Done (t=4.06s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=36.54s).
Accumulating evaluation results...
DONE (t=5.35s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.001
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.002
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.001
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.001
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.001
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

In [9]:
def generate_metrics_dict(cocoEval):
    keys = [
        "AP_IoU_0.50:0.95_all_maxDets_100",
        "AP_IoU_0.50_all_maxDets_100",
        "AP_IoU_0.75_all_maxDets_100",
        "AP_IoU_0.50:0.95_small_maxDets_100",
        "AP_IoU_0.50:0.95_medium_maxDets_100",
        "AP_IoU_0.50:0.95_large_maxDets_100",
        "AR_IoU_0.50:0.95_all_maxDets_1",
        "AR_IoU_0.50:0.95_all_maxDets_10",
        "AR_IoU_0.50:0.95_all_maxDets_100",
        "AR_IoU_0.50:0.95_small_maxDets_100",
        "AR_IoU_0.50:0.95_medium_maxDets_100",
        "AR_IoU_0.50:0.95_large_maxDets_100"
    ]
    metrics = {key: cocoEval.stats[i] for i, key in enumerate(keys)}
    return metrics

metrics = generate_metrics_dict(cocoEval)
metrics

{'AP_IoU_0.50:0.95_all_maxDets_100': 0.0013413949568033725,
 'AP_IoU_0.50_all_maxDets_100': 0.002164254887027164,
 'AP_IoU_0.75_all_maxDets_100': 0.0014215484048404837,
 'AP_IoU_0.50:0.95_small_maxDets_100': 0.00022806697463059373,
 'AP_IoU_0.50:0.95_medium_maxDets_100': 0.0009267739273927392,
 'AP_IoU_0.50:0.95_large_maxDets_100': 0.000655940594059406,
 'AR_IoU_0.50:0.95_all_maxDets_1': 6.766536086270842e-05,
 'AR_IoU_0.50:0.95_all_maxDets_10': 0.00013464758563735732,
 'AR_IoU_0.50:0.95_all_maxDets_100': 0.00013902903592579843,
 'AR_IoU_0.50:0.95_small_maxDets_100': 4.241130320800951e-05,
 'AR_IoU_0.50:0.95_medium_maxDets_100': 0.00018514039856088366,
 'AR_IoU_0.50:0.95_large_maxDets_100': 0.00014938312721336043}